In [ ]:
import os
os.chdir('???')
os.getcwd()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
df = iris.frame
class_names = iris.target_names
print(class_names)
df.head()

In [ ]:
output_var = 'target'
input_vars = df.columns.tolist()
input_vars.remove(output_var)
input_vars

In [ ]:
X = df[input_vars]
y = df[output_var] 
X.head()

### Linear Discriminant Analysis

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
selected_n_components = 2 # can be adjusted
lda = LinearDiscriminantAnalysis(n_components=selected_n_components)

X_lda = lda.fit_transform(X_scaled, y)

In [ ]:
target_names = class_names
plt.figure()
for label, name in enumerate(target_names):
    plt.scatter(
        X_lda[y == label, 0],
        X_lda[y == label, 1],
        label=name
    )

plt.xlabel("LD1")
plt.ylabel("LD2")
plt.title("LDA")
plt.legend()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

y = df[output_var]
var1_index = 0
var2_index = 1

# Left figure: X=var1, Y = var2
axs[0].scatter(X_scaled[:, var1_index], X_scaled[:, var2_index], c=y, alpha=0.6)
axs[0].set_title("X scaled")
axs[0].set_xlabel('Scaled '+input_vars[var1_index])
axs[0].set_ylabel('Scaled '+input_vars[var2_index])
axs[0].axis("equal")

# Left figure: X=LDA1, Y = LDA2
scatter2 = axs[1].scatter(X_lda[:, var1_index], X_lda[:, var2_index], c=y, alpha=0.6)
axs[1].set_title("X LDA-transformed")
axs[1].set_xlabel("LDA1")
axs[1].set_ylabel("LDA2")

plt.tight_layout()
plt.show()

In [ ]:
# How different variables contributes class separation
print("Discriminant coefficients (w):")
print(lda.scalings_)

In [ ]:
# Interpret variable contribution to class separation
# Larger absolute value = stronger contribution to class separation
# Direction indicates which class side
# Compare coefficients within the same discriminant axis

scalings_df = pd.DataFrame(
    lda.scalings_,
    index=X.columns,
    columns=[f"LD{i+1}" for i in range(lda.scalings_.shape[1])]
)

scalings_df.abs().sort_values(by="LD1", ascending=False)

In [ ]:
# LDA scaling can be "roughly" used to rank variables by magnitude
# Note: To get exact and better ranking, perform feature selection instead.  
#       LDA is mainly used to find projection for best separation, not for feature importance.

feature_importance = scalings_df.abs().sum(axis=1)
feature_importance.sort_values(ascending=False)

In [ ]:
# Classifier coefficients
# Magnitude of coefficients: Which  variable is important for which class
# Sign of coefficients: Direction of separation
coef_df = pd.DataFrame(
    lda.coef_,
    columns=input_vars,
    index=class_names
)
coef_df

# Result interpretation
# Setosa: small petal length & width
# Virginica: large petal length & width
# Versicolor: intermediate

In [ ]:
# LDA Explained Variance Plot
# LDA bar height = proportion of between-class variance explained
# Note: PCA bar height = total data variance != LDA bar height 

plot_y = [val * 100 for val in lda.explained_variance_ratio_]
plot_x = range(1, len(plot_y) + 1)

bars = plt.bar(plot_x, plot_y, align="center", color="#1C3041", edgecolor="#000000", linewidth=1.2)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 0.001, f"{yval:.1f}%", ha="center", va="bottom")

plt.xlabel("Linear Discriminant")
plt.ylabel("Percentage of Explained Variance")
plt.title("Variance Explained per Linear Discriminant", loc="left", fontdict={"weight": "bold"}, y=1.06)
plt.grid(axis="y")
plt.xticks(plot_x)

plt.show()

In [ ]:
# Create a Grid of Points
temp_X = X.iloc[:, :2]   # first two features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
temp_X_scaled = scaler.fit_transform(temp_X)
temp_lda = LinearDiscriminantAnalysis()
temp_lda.fit(temp_X_scaled, y)

x_min, x_max = temp_X_scaled[:, 0].min() - 0.5, temp_X_scaled[:, 0].max() + 0.5
y_min, y_max = temp_X_scaled[:, 1].min() - 0.5, temp_X_scaled[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)
grid = np.c_[xx.ravel(), yy.ravel()]

In [ ]:
# plot Decision regions

var1_index = 0
var2_index = 1

Z = temp_lda.predict(grid)
Z = Z.reshape(xx.shape)
plt.contourf(xx, yy, Z, alpha=0.3)

# Training points
for class_label in np.unique(y):
    plt.scatter(
        temp_X_scaled[y == class_label][:, var1_index],
        temp_X_scaled[y == class_label][:, var2_index],
        label=class_label
    )

plt.xlabel('Scaled '+input_vars[var1_index])
plt.ylabel('Scaled '+input_vars[var2_index])
plt.title("LDA Classification Decision Boundaries")
plt.legend()
plt.show()

### Principal Component Analysis 

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure()
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y)
plt.title("PCA")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

y = df[output_var]
var1_index = 0
var2_index = 1

# Left figure: X=var1, Y = var2
axs[0].scatter(X_scaled[:, var1_index], X_scaled[:, var2_index], c=y, alpha=0.6)
axs[0].set_title("X scaled")
axs[0].set_xlabel(input_vars[var1_index])
axs[0].set_ylabel(input_vars[var2_index])
axs[0].axis("equal")

# Left figure: X=PC1, Y = PC2
scatter2 = axs[1].scatter(X_pca[:, var1_index], X_pca[:, var2_index], c=y, alpha=0.6)
axs[1].set_title("X PCA-transformed")
axs[1].set_xlabel("PC1")
axs[1].set_ylabel("PC2")

plt.tight_layout()
plt.show()

In [ ]:
plot_y = [val * 100 for val in pca.explained_variance_ratio_]
plot_x = range(1, len(plot_y) + 1)

bars = plt.bar(plot_x, plot_y, align="center", color="#1C3041", edgecolor="#000000", linewidth=1.2)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 0.001, f"{yval:.1f}%", ha="center", va="bottom")

plt.xlabel("Principal Component")
plt.ylabel("Percentage of Explained Variance")
plt.title("Variance Explained per Principal Component", loc="left", fontdict={"weight": "bold"}, y=1.06)
plt.grid(axis="y")
plt.xticks(plot_x)

plt.show()

In [ ]:
exp_var = [val * 100 for val in pca.explained_variance_ratio_]
plot_y = [sum(exp_var[:i+1]) for i in range(len(exp_var))]
plot_x = range(1, len(plot_y) + 1)

plt.plot(plot_x, plot_y, marker="o", color="#9B1D20")
for x, y in zip(plot_x, plot_y):
    plt.text(x, y + 1.5, f"{y:.1f}%", ha="center", va="bottom")

plt.xlabel("Principal Component")
plt.ylabel("Cumulative Percentage of Explained Variance")
plt.title("Cumulative Variance Explained per Principal Component", loc="left", fontdict={"weight": "bold"}, y=1.06)

plt.yticks(range(0, 101, 5))
plt.grid(axis="y")
plt.xticks(plot_x)

plt.show()

In [ ]:
# To see contribution of each original variable on each PC
# Larger absolute value = stronger contribution to that PC
# Direction (+/-) shows correlation direction
# Meaningful within the same PC
# Do not use this to look at importance of variables

components_df = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)
components_df

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(components_df,cmap='coolwarm')
plt.show()

# y axis = each variable
# The result shows that each variable contributes to each principal component